# Task 2.1: Dataset Selection and Setup

### Dataset Choice
We are using a synthetic dataset generated by `sklearn.datasets.make_regression`. We requested 1,000 samples and 100 features, but critically, we set `n_informative=10`. This means that $90\%$ of the dataset features are pure noise and completely irrelevant to predicting the target variable.

### Why is this appropriate?
This dataset is an ideal, controlled testbed for the "Truncated Gradient" (TG) method. The central claim in the Langford et al. paper is that Truncated Gradient actively induces sparsity by shrinking small, irrelevant weights exactly to zero, which is exceptionally useful for "large datasets... with large numbers of features, substantial sparsity is discoverable." By artificially creating a dataset where exactly 10 features out of 100 matter, we can precisely evaluate whether TG correctly drives the 90 irrelevant feature weights to exactly $0.0$, and we can compare this directly against standard Gradient Descent.

### Limitations compared to original paper
While this dataset mimics the *proportion* of sparsity found in real-world large-scale data, it lacks the sheer scale (the paper used datasets with millions of samples and billions of features like `Big_Ads`). Furthermore, synthetic Gaussian noise features behave perfectly mathematically, unlike the complex correlated noise, heteroscedasticity, or missing data structures commonly found in actual ad-click distributions or text-categorization (`rcv1`) sets evaluated in the original paper.

In [1]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import os

# Set seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Ensure data directory exists
os.makedirs('data', exist_ok=True)

# 1. Generate the dataset
X, y, coef = make_regression(
    n_samples=1000,
    n_features=100,
    n_informative=10,
    noise=1.0,           
    coef=True,           # Return the underlying true coefficients so we know the "ground truth" informative ones
    random_state=RANDOM_SEED
)

# 2. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

# 3. Preprocessing (Standard Scaling is critical for Gradient Descent / Square Loss stability)
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.reshape(-1, 1)).flatten()
y_test_scaled = scaler_y.transform(y_test.reshape(-1, 1)).flatten()

# Save Data
np.save('data/X_train.npy', X_train_scaled)
np.save('data/X_test.npy', X_test_scaled)
np.save('data/y_train.npy', y_train_scaled)
np.save('data/y_test.npy', y_test_scaled)
np.save('data/true_coefficients.npy', coef)

with open('data/README.md', 'w') as f:
    f.write("Dataset generated using sklearn make_regression with 1000 samples, 100 features, and 10 informative features. Data is StandardScaled and saved as numpy arrays.")

print(f"Dataset generated: {X_train_scaled.shape[0]} training samples, {X_test_scaled.shape[0]} test samples, {X_train_scaled.shape[1]} features.")

Dataset generated: 800 training samples, 200 test samples, 100 features.
